# Assumption Diagnostic Notes

Coverage survey for regression assumption diagnostics, organized by the seven assumptions
taught in DATA 5600 (lectures 06, 07, and 14).

## Why by assumption, not by package

The rows are fixed by what the course actually teaches. That keeps the survey anchored to
what students are asked to do in Exercises 06, 07, and 14, rather than to whatever
`performance` happens to implement.

## Columns

| Column | Meaning |
|---|---|
| **Tool** | Package + function |
| **Call** | Minimal invocation |
| **Covers** | Does it address this assumption? |
| **Misuse risk** | How easy is it to get a plausible-but-wrong answer? |
| **Verified** | Have we run it against the course data ourselves? |

`Covers`: OK = yes / PARTIAL = partial / NO = no / ? = unevaluated
`Misuse risk`: Low / Med / High / ?

## Why "misuse risk" is a column

Coverage alone is misleading. `statsmodels.variance_inflation_factor` exists, is the
standard tool, and is correctly documented -- and both the lecture (07) and the prior
prototype got it wrong the same way, because the API accepts a bare predictor matrix
without complaint and returns plausible numbers. See the Identifiability section.

The contribution of a course library is probably not reimplementing these functions.
It is making the wrong call impossible.


## At a glance

| # | Assumption | Course diagnostic | Python coverage | Biggest gap |
|---|---|---|---|---|
| 1 | Validity | qualitative only, no code | **OK -- ~5 lines, no new deps** | course never implements it; PPC machinery sits unused in L16 |
| 2 | Representativeness | DFFITS + threshold line | **PARTIAL -- OLS good, GLM broken** | methods disagree (0/4 consensus); `influence_plot` raises on GLM |
| 3 | Linearity | 3 scatterplots + partial regression grid | ? | **logistic log-odds check has no code** |
| 4 | Independence | residuals vs. row order | ? | no formal test taught |
| 5 | Constant variance *(linear only)* | residuals vs. fitted | ? | no formal test taught |
| 6 | Normality *(linear only)* | residual hist + Q-Q | ? | no calibration for "bad enough" |
| 7 | Identifiability | corr matrix + VIF | ? | **VIF intercept trap; no GVIF** |

Sections 1 and 2 surveyed and verified against the course data. 3-7 pending.


In [1]:
# Reference models the survey runs against.
# Linear: soft-launch data, prep follows Exercise 10. VIF setup corrected (see Identifiability).
# Logistic: real leads data (replaces the prior prototype's synthetic "units above median" flag).

import os
import numpy as np, polars as pl, pandas as pd
import statsmodels.api as sm, statsmodels.formula.api as smf

brand_names = {'JIF':'Jif','Jiff':'Jif','SKIPPY':'Skippy','Skipy':'Skippy',
               'Skipp':'Skippy','Peter Pan':'PeterPan',"Harmon's":'Harmons'}

pb = (pl.read_parquet(os.path.join('data','soft_launch.parquet'))
    .with_columns(pl.col('brand').cast(pl.Utf8).replace(brand_names).alias('brand'))
    .drop_nans(['units','sales'])
    .remove((pl.col('brand')=="None") | (pl.col('loyal')=="None") |
            (pl.col('texture')=="None") | (pl.col('size')=="None"))
    .with_columns([pl.col('price').cast(pl.Float64),
                   pl.col('loyal').cast(pl.Int64),
                   pl.col('promo').cast(pl.Int64)])
    .unique()
    .filter(pl.col('units') > 0))

pb_train = (pb
    .with_columns([(pl.col('units')+1).log().alias('log_units'),
                   (pl.col('price')+1).log().alias('log_price')])
    .to_dummies(columns=['brand','texture','size'], drop_first=False)
    .select(pl.exclude('units','price','brand_Jif','texture_Smooth','size_16'))
    .to_pandas())

predictors = ['brand_Harmons','brand_PeterPan','brand_Skippy','coupon','ad',
              'texture_Chunky','size_12','log_price']

fit = smf.ols('log_units ~ ' + ' + '.join(predictors), data=pb_train).fit()

# Design matrix WITH intercept -- required by variance_inflation_factor.
X_vif = sm.add_constant(pb_train[predictors])

# --- Logistic reference model: real leads data, balanced 235/235 by Marc's downsampling ---
leads = (pl.read_parquet(os.path.join('data','leads.parquet'))
    .with_columns((pl.col('Stage') == "Qualified").cast(pl.Int64).alias('qualified'))
    .to_pandas()
    .rename(columns={'ActivityTypePhone Call':'phone',
                     'ActivityTypeEmail':'email',
                     'ActivityTypeMeeting':'meeting'}))

glm = smf.glm('qualified ~ days_elapsed + email + phone + meeting',
              data=leads, family=sm.families.Binomial()).fit()

print(f"OLS  n={int(fit.nobs):5d}  p={len(fit.params)}")
print(f"GLM  n={int(glm.nobs):5d}  p={len(glm.params)}  converged={glm.converged}")


FileNotFoundError: The system cannot find the path specified. (os error 3): data\soft_launch.parquet

## 1. Validity

*Data is relevant to the objective and no predictors are missing.*

**Course (L06, L14):** qualitative questions only -- is the data relevant, does it match the
ideal data, how do simulated and real outcomes compare. **No code is provided.**

| Tool | Call | Covers | Misuse risk | Verified |
|---|---|---|---|---|
| *(none in course)* | -- | NO | -- | -- |
| **simulated-outcome check** | `rng.normal(fit.fittedvalues, sqrt(fit.scale))` | OK | Low | **yes -- ~5 lines, no Bayesian fit** |
| `sm` `get_prediction` | `fit.get_prediction()` | NO | Med | gives intervals for E[y], not replicate datasets |
| `arviz.plot_ppc` | needs a Bayesian fit | OK | Low | works, but unnecessary -- see below |
| R `performance::check_predictions()` | observed vs. model-simulated density | -- | -- | reference implementation |

### Verified: this needs no Bayesian machinery

The course frames "how do the simulated and real outcomes compare" as a prose question, and
teaches posterior predictive checks only in L16 with `bambi`/`arviz`. But for OLS the check is
a parametric simulation from the fitted model -- it needs `fit.fittedvalues` and `fit.scale`
and nothing else:

```python
y_rep = rng.normal(fit.fittedvalues.values[:, None], np.sqrt(fit.scale), size=(n, B))
```

That is the whole implementation. `performance::check_predictions()` does exactly this for `lm`.
No MCMC, no extra dependency. **Cheapest win in the survey, and cheaper than first assumed.**

### It immediately finds a real problem in the course's own model

Run against the flagship soft-launch OLS (`log_units ~ ...`, n = 1,095, B = 50, seed 42):

| Statistic | Observed | Replicate range | Percentile |
|---|---|---|---|
| min | 1.099 | [0.513, 0.959] | **1.00** |
| max | 2.996 | [2.837, 3.752] | 0.24 |
| mean | 1.857 | [1.847, 1.871] | 0.54 |
| sd | 0.364 | [0.352, 0.378] | 0.40 |
| skew | 0.246 | [0.033, 0.326] | 0.94 |

Mean and sd match well -- the model gets the centre right. The observed **minimum** sits outside the entire replicate range. Cause: `units` is a count with
a floor of 2, so `log(units + 1)` cannot go below `log(3) = 1.099`. The normal likelihood has no
such floor and happily simulates values beneath it.

So the model is misspecified in a way the course never surfaces -- a continuous, unbounded
likelihood on a discrete, bounded outcome. This is precisely what a predictive check is for, and
it falls out of the least expensive diagnostic on the list.

**Implication for scoping:** validity goes from "no code exists" to "5 lines, high payoff." It
should probably lead the port rather than be treated as the qualitative leftover.


In [ ]:
# 1. Validity -- simulated-outcome check (no Bayesian fit required)
import numpy as np, pandas as pd

def check_predictions(fit, B=50, seed=42):
    """Replicate datasets from a fitted OLS. Frequentist analogue of a PPC."""
    rng = np.random.default_rng(seed)
    n = int(fit.nobs)
    y_rep = rng.normal(fit.fittedvalues.values[:, None], np.sqrt(fit.scale), size=(n, B))
    return fit.model.endog, y_rep

def ppc_stats(y_obs, y_rep):
    rows = []
    for name, f in [("min", np.min), ("max", np.max),
                    ("mean", np.mean), ("sd", np.std),
                    ("skew", lambda a: pd.Series(a).skew())]:
        obs = f(y_obs)
        rep = np.array([f(y_rep[:, b]) for b in range(y_rep.shape[1])])
        rows.append({"stat": name, "observed": obs,
                     "rep_lo": rep.min(), "rep_hi": rep.max(),
                     "pct": (rep < obs).mean()})
    return pd.DataFrame(rows)

y_obs, y_rep = check_predictions(fit)
print(ppc_stats(y_obs, y_rep).round(3).to_string(index=False))
# pct == 1.00 or 0.00 means observed is outside the replicate range entirely.


## 2. Representativeness

*Data is representative of the data generating process or population.*

**Course (L06):** DFFITS via `fit.get_influence().dffits[0]`, plotted against observation number
with a $2\sqrt{p/n}$ threshold line. Remedy framing is explicitly *investigate, don't delete*.

| Tool | Call | Covers | Misuse risk | Verified |
|---|---|---|---|---|
| `sm` OLSInfluence | `fit.get_influence().summary_frame()` | OK | Low | **yes -- 15 cols in one call** |
| `sm` GLMInfluence | `glm.get_influence().summary_frame()` | **PARTIAL** | **High** | **yes -- missing `dffits` + `student_resid`** |
| `sm` MLEInfluence | -- | -- | -- | **resolved: base class of GLMInfluence, not a separate option** |
| `sm.graphics.influence_plot` | `influence_plot(fit, criterion='dffits')` | PARTIAL | **High** | **yes -- works on OLS, raises on GLM** |
| R `performance::check_outliers()` | ~15 methods + composite | -- | -- | reference implementation |

### Verified: one call replaces the course's manual DFFITS block

`summary_frame()` on the soft-launch OLS returns 15 columns for n = 1,095: `dfb_*` per
coefficient, `cooks_d`, `standard_resid`, `hat_diag`, `dffits_internal`, `student_resid`,
`dffits`. The course computes one of these by hand.

### Verified: the methods disagree almost completely

Same model, four standard rules:

| Rule | Flagged |
|---|---|
| `|dffits| > 2*sqrt(p/n)` | 60 |
| `cooks_d > 4/n` | 59 |
| `hat_diag > 2p/n` | 72 |
| `|student_resid| > 3` | 5 |
| **flagged by >= 2 rules** | **59** |
| **flagged by all 4** | **0** |

Not one observation is flagged by every rule. The course teaches a single measure with a single
threshold and presents the result as the answer -- but which points you "find" is mostly an
artifact of which rule you were taught. This is the argument for `check_outliers()`-style
composite logic, and it is now empirically grounded rather than asserted.

### Verified: GLMInfluence is not API-compatible with OLSInfluence

`GLMInfluence.summary_frame()` returns **9** columns, not 15 -- `dffits` and `student_resid`
are absent, and there is no `.dffits` attribute. Consequence:

```python
sm.graphics.influence_plot(ols_fit, criterion='dffits')   # OK
sm.graphics.influence_plot(glm_fit, criterion='dffits')   # AttributeError
```

The same statsmodels function, with the default argument, silently works for linear regression
and crashes for logistic. A student following L06's approach into Exercise 14 hits this.

**This is a first-class finding for the library:** a uniform influence interface across families
is real value, not a convenience wrapper.


In [ ]:
# 2. Representativeness -- influence measures, method agreement, GLM asymmetry
import numpy as np, pandas as pd, statsmodels.api as sm

# --- OLS: everything in one call ---
sf = fit.get_influence().summary_frame()
print("OLSInfluence.summary_frame():", sf.shape, "\n", list(sf.columns), "\n")

n, p = fit.nobs, len(fit.params)
rules = {
    "dffits":        sf["dffits"].abs()  > 2*np.sqrt(p/n),
    "cooks_d":       sf["cooks_d"]       > 4/n,
    "hat_diag":      sf["hat_diag"]      > 2*p/n,
    "student_resid": sf["student_resid"].abs() > 3,
}
agree = sum(r.astype(int) for r in rules.values())
print(pd.Series({k: int(v.sum()) for k, v in rules.items()}).to_string())
print(f">=2 rules: {(agree>=2).sum()}   all 4: {(agree==4).sum()}")

# --- GLM: NOT the same interface ---
gi = glm.get_influence()
gsf = gi.summary_frame()
print("\nGLMInfluence.summary_frame():", gsf.shape)
# compare structural columns only -- dfb_* differ because the models have different predictors
structural = lambda cols: {c for c in cols if not c.startswith("dfb_")}
print("missing vs OLS:", sorted(structural(sf.columns) - structural(gsf.columns)))
print("has .dffits attribute:", hasattr(gi, "dffits"))

# influence_plot: same call, different outcome
for name, m in [("OLS", fit), ("GLM", glm)]:
    try:
        sm.graphics.influence_plot(m, criterion="dffits")
        print(f"influence_plot({name}): OK")
    except AttributeError as e:
        print(f"influence_plot({name}): AttributeError -- {e}")


## 3. Linearity

*The mapping function is an additive, linear function of the parameters.*

**Course (L06):** scatter of $y$ vs. $x$, residuals vs. $x$, residuals vs. fitted, plus `plot_partregress_grid`.

**Course (L14, logistic):** scatter of **log-odds** vs. predictors -- presented as `figures/temp_linearity.png`, a static image. **There is no code for this anywhere in the repo**, yet Exercise 14 asks students to run it.

| Tool | Call | Covers | Misuse risk | Verified |
|---|---|---|---|---|
| `sm.graphics.plot_partregress_grid` | `(fit)` | OK | ? | in course |
| `sm.graphics.plot_regress_exog` | `(fit, 'x')` | OK | ? | 4-panel per predictor; **not used by course** |
| `sm.graphics.plot_ccpr_grid` | `(fit)` | ? | ? | no |
| `sm.graphics.plot_ceres_residuals` | `(fit, 'x')` | ? | ? | no |
| `sm` `linear_reset` | `linear_reset(fit)` | ? | ? | exists |
| `sm` `linear_rainbow` | `linear_rainbow(fit)` | ? | ? | exists |
| `sm` `linear_harvey_collier` | `linear_harvey_collier(fit)` | ? | Med? | exists; assumes meaningful ordering |
| binned residuals *(logistic)* | port from prior prototype | PARTIAL | ? | matches `performance::binned_residuals()` |
| `lmdiag` | `lmdiag.plot(fit)` | ? | ? | **not evaluated** |

**Priority gap:** the one diagnostic unique to logistic regression is the one with no implementation.


In [ ]:
# scratch: 3. Linearity


## 4. Independence

*Observations are independent (iid, or at least exchangeable).*

**Course (L07):** plot residuals against row order; look for trend in mean or variance. The lecture then asks *"do we need to check for sequential trends with this data?"* -- the peanut butter data has no natural sequence.

| Tool | Call | Covers | Misuse risk | Verified |
|---|---|---|---|---|
| `sm` `durbin_watson` | `durbin_watson(fit.resid)` | PARTIAL | **High** | only meaningful with real ordering |
| `sm` `acorr_ljungbox` | `acorr_ljungbox(fit.resid, lags=[5])` | PARTIAL | **High** | same caveat |
| `sm` `acorr_breusch_godfrey` | `acorr_breusch_godfrey(fit, nlags=5)` | PARTIAL | **High** | same caveat |

**Open question:** independence is mostly a *design* question (clustering, repeat measures),
not a residual question. Running these on unordered cross-sectional data produces a number
that means nothing. Does the library refuse to compute them without a declared ordering?


In [ ]:
# scratch: 4. Independence


## 5. Constant Variance *(linear only)*

*Homoscedasticity of errors.*

**Course (L07):** scatter of residuals vs. fitted; look for a funnel. Struck out for logistic in L14.

| Tool | Call | Covers | Misuse risk | Verified |
|---|---|---|---|---|
| `sm` `het_breuschpagan` | `het_breuschpagan(fit.resid, fit.model.exog)` | OK | Med | over-flags at large n |
| `sm` `het_white` | `het_white(fit.resid, fit.model.exog)` | OK | Med | very sensitive at n~4700 |
| `sm` `het_goldfeldquandt` | -- | ? | ? | no |
| dispersion ratio *(GLM)* | `(resid_pearson**2).sum()/df_resid` | PARTIAL | ? | GLM analogue |
| R `performance::check_heteroscedasticity()` | -- | -- | -- | reference |

**Note:** at n = 4,700 these tests reject on trivial departures. This is exactly why the course
teaches eyeballs. A library that reports a bare p-value here makes things *worse* -- see
Cross-Cutting / reference bands.


In [ ]:
# scratch: 5. Constant Variance *(linear only)*


## 6. Normality *(linear only)*

*Variation or error is normally distributed -- **not** that the outcome is.*

**Course (L07):** residual histogram + `sm.qqplot(resid, line='45', fit=True)`. Struck out for logistic in L14.

| Tool | Call | Covers | Misuse risk | Verified |
|---|---|---|---|---|
| `sm.qqplot` | `sm.qqplot(fit.resid, line='45')` | OK | Low | in course |
| `sm` `normal_ad` | `normal_ad(fit.resid)` | OK | Med | Anderson-Darling |
| `sm` `lilliefors` | `lilliefors(fit.resid)` | ? | Med | exists |
| `scipy.stats.shapiro` | `st.shapiro(fit.resid)` | OK | **High** | over-flags badly at large n |
| `scipy.stats.probplot` | -- | OK | Low | -- |
| randomized quantile resid *(GLM)* | port from prior prototype | OK | ? | verified correct for Bernoulli |
| R `DHARMa` | simulated quantile residuals | -- | -- | reference |

**Watch:** the existing helper `topics/14_logistic-regression/model_diagnostics.py` checks the
distribution of **y**, not the residuals -- contradicting L07's explicit statement. It is
imported nowhere. Decide whether to fix or retire it.


In [ ]:
# scratch: 6. Normality *(linear only)*


## 7. Identifiability

*Parameters can be estimated; no strong multicollinearity.*

**Course (L07, L14):** scatterplot matrix, correlation matrix, VIF via a loop over `variance_inflation_factor`. Rule of thumb: largest VIF much more than 10, or mean much more than 1.

| Tool | Call | Covers | Misuse risk | Verified |
|---|---|---|---|---|
| `sm` `variance_inflation_factor` | `vif(sm.add_constant(X).values, i)` | OK | **High** | yes -- see below |
| correlation matrix | `X.corr()` | PARTIAL | Low | in course |
| GVIF for categoricals | -- | **NO** | -- | **confirmed absent from statsmodels** |
| R `car::vif()` | returns GVIF + GVIF^(1/2k) | -- | -- | reference implementation to port |

### The intercept trap (verified)

`variance_inflation_factor` expects the design matrix **including the constant**. Lecture 07
and the prior prototype both pass predictors only. Same model, same data:

| Predictor | VIF without intercept | VIF with intercept |
|---|---|---|
| `brand_Harmons` | 1.48 | **81.86** |
| `size_12` | 1.34 | **17.49** |
| `log_price` | 2.88 | **95.20** |

Without the constant: every VIF under 3, "no multicollinearity." With it: catastrophic.
The corrected numbers are right -- the simulation builds `price = price_per_oz[brand] * size`,
so price *is* a function of brand and size by construction.

The broken version misses the one structural feature deliberately built into the data.

### GVIF

`brand` is 4 dummy columns. Reporting 4 separate VIFs for one conceptual predictor is what
GVIF fixes: for a term with $k$ columns, $\\text{GVIF} = \\frac{\\det(R_{11})\\det(R_{22})}{\\det(R)}$,
reported on the comparable scale as $\\text{GVIF}^{1/(2k)}$. Small to implement; nothing in
Python has it.


In [ ]:
# scratch: 7. Identifiability


## Cross-cutting requirements

These are **not** per-assumption and **not** search questions -- nothing in Python provides
them, so they are design decisions.

| Requirement | Exists in Python? | Notes |
|---|---|---|
| Single entry point (`check_model()`) | **No** | API design: one call, dispatch on model family |
| Plain-language verdicts | **No** | string templates keyed to test + threshold |
| Result objects that print *and* plot | **No** | `__repr__` + `.plot()` |
| **Simulation-based reference bands** | **No** | see below |

### Reference bands -- the highest-value item

The failure mode of visual diagnostics is that students cannot calibrate them. Shown a Q-Q
plot with tail curvature, they have no way to answer *"is this bad enough?"*

`performance::check_model()` simulates from the fitted model and draws a band showing what a
correctly-specified model looks like **at that sample size**. The judgment becomes
"inside or outside the band."

This also fixes the large-n problem in sections 5 and 6: instead of a p-value that always
rejects at n=4,700, the band shows the actual magnitude of departure.

The machinery is mostly present -- the quantile-residual function plus B simulated replicates
and pointwise quantiles.


## Package-level notes

**scikit-learn** -- effectively no coverage. No residual accessors, no influence measures, no
VIF, no formal tests; `LinearRegression` does not expose standard errors. It is a prediction
library. One line in the writeup, not a section.

**statsmodels** -- the workhorse, but the surface is scattered across four submodules
(`stats.diagnostic`, `stats.outliers_influence`, `stats.stattools`, `graphics.regressionplots`).
Discoverability is itself part of the problem being solved.

**Unevaluated, worth ~20 min each:** `pingouin` (`normality()`, `homoscedasticity()`),
`lmdiag` v0.4.1 (R `plot.lm` four-panel), `yellowbrick` (`ResidualsPlot`, `CooksDistance`).

**`rpy2` as an oracle** -- not the deliverable, but calling `performance::check_model()` on the
same data gives an independent reference to validate against. Given that this project started
from a bug that survived two authors, that is worth the setup cost.


## Open scope decisions

1. **Which model families?** The course uses OLS, logistic, ridge/LASSO/elastic net, and PCR.
   Penalized models have no standard residual theory, no `get_influence()`, and shrunk
   coefficients make VIF ill-defined. PCR forces all VIFs to 1 by construction -- L21 says so.
   `performance` handles `glmnet` poorly and has no PCR story. Scope cut, or the novel part?

   *Working proposal: 7 assumptions x 2 families (OLS, logistic) for the core, with a
   documented position on penalized/PCR.*

2. **Port vs. teaching library?** `performance` is ~20k lines across dozens of model classes.
   The course needs ~8 checks on 2 families.

3. **Retire or fix `model_diagnostics.py`?** It exists in topic 14, is imported nowhere, and
   its normality check contradicts L07.

4. **Does the library refuse bad calls?** e.g. no Durbin-Watson without a declared ordering;
   VIF that adds the constant itself. This is where "misuse risk" becomes a design principle.
